# NB05 · Demostrador de inferencia para palma africana

**Entorno recomendado:** CPU, sin acelerador  
**GPU:** opcional; el notebook la utiliza automáticamente si está disponible  
**Tiempo estimado:** 5–10 minutos, incluida la instalación; después, pocos segundos por lote

Este notebook carga el modelo final `YOLO11m · semilla 2026` y permite:

1. subir fotografías nuevas desde el computador;
2. clasificar cada imagen en cinco categorías;
3. visualizar los cinco puntajes del modelo;
4. identificar decisiones poco separadas mediante una regla didáctica;
5. guardar tablas y figuras en Drive para el informe y el libro.

Clases: `damaged`, `empty`, `overripe`, `ripe` y `unripe`.

> El sistema clasifica la imagen completa. La fotografía debe contener un racimo principal, visible y centrado. No localiza ni cuenta racimos.


In [ ]:
%pip install -q -U ultralytics


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import math
import platform
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from PIL import Image, ImageOps
from ultralytics import YOLO

PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
MODEL_PATH = PROJECT_ROOT / '05_MODELOS' / 'yolo11m_cls_seed2026' / 'best.pt'
RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS' / 'NB05_DEMOSTRADOR_INFERENCIA'
BOOK_ROOT = PROJECT_ROOT / '08_MATERIAL_LIBRO' / 'NB05_DEMOSTRADOR_INFERENCIA'
UPLOAD_DIR = Path('/content/palma_imagenes_subidas')

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_RESULTS = RESULTS_ROOT / RUN_ID
RUN_BOOK = BOOK_ROOT / RUN_ID
ORIGINALS_DIR = RUN_RESULTS / 'imagenes_originales'
FIGURES_DIR = RUN_BOOK / 'figuras_individuales'

for path in [RUN_RESULTS, RUN_BOOK, ORIGINALS_DIR, FIGURES_DIR, UPLOAD_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'No se encontró el modelo final: {MODEL_PATH}. Ejecute NB04.')

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
SPANISH = {
    'damaged': 'dañado',
    'empty': 'vacío',
    'overripe': 'sobremaduro',
    'ripe': 'maduro',
    'unripe': 'inmaduro',
}
COLORS = {
    'damaged': '#b2182b',
    'empty': '#8c510a',
    'overripe': '#d6604d',
    'ripe': '#4d9221',
    'unripe': '#2166ac',
}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

print('Dispositivo:', DEVICE_NAME)
print('Modelo:', MODEL_PATH)
print('Ultralytics:', ultralytics.__version__)
print('Resultados:', RUN_RESULTS)
print('Material para el libro:', RUN_BOOK)


In [ ]:
# Verificación rápida del modelo antes de solicitar imágenes.
model = YOLO(str(MODEL_PATH))
model_names = [model.names[i] for i in sorted(model.names)]
if model_names != CLASSES:
    raise RuntimeError(f'Las clases del modelo no coinciden. Encontradas: {model_names}')

print('Modelo cargado correctamente.')
print('Clases:', model_names)


## Subir imágenes

Ejecute la siguiente celda y seleccione una o varias fotografías. Se admiten JPG, JPEG, PNG, BMP, TIFF y WEBP.

Las imágenes subidas se copiarán dentro de la carpeta de resultados de esta ejecución en su propio Drive.


In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No se seleccionaron imágenes. Vuelva a ejecutar esta celda.')

uploaded_paths = []
for filename, data in uploaded.items():
    safe_name = Path(filename).name
    destination = UPLOAD_DIR / safe_name
    destination.write_bytes(data)
    if destination.suffix.lower() not in IMAGE_EXTS:
        print('Omitida por formato no compatible:', safe_name)
        continue
    try:
        with Image.open(destination) as im:
            im.verify()
        uploaded_paths.append(destination)
        shutil.copy2(destination, ORIGINALS_DIR / safe_name)
    except Exception as exc:
        print(f'Omitida {safe_name}: {exc}')

uploaded_paths = sorted(set(uploaded_paths))
if not uploaded_paths:
    raise RuntimeError('Ninguno de los archivos seleccionados es una imagen válida.')

print('Imágenes válidas:', len(uploaded_paths))
for path in uploaded_paths:
    print('-', path.name)


## Predicción y regla de revisión

Los valores mostrados son puntajes `softmax`, no probabilidades clínicamente calibradas. Para fines didácticos se utiliza una regla de separación:

- **decisión clara:** puntaje máximo ≥ 0,75 y margen entre las dos primeras clases ≥ 0,25;
- **revisar:** cualquier otro caso.

Esta regla sirve para señalar imágenes ambiguas, pero no sustituye una validación agronómica ni convierte el sistema en diagnóstico.


In [ ]:
start = time.perf_counter()
outputs = model.predict(
    source=[str(path) for path in uploaded_paths],
    imgsz=384,
    batch=min(16, len(uploaded_paths)),
    device=DEVICE,
    verbose=False,
)
elapsed = time.perf_counter() - start

rows = []
for path, result in zip(uploaded_paths, outputs):
    probs = result.probs.data.detach().cpu().numpy().astype(float)
    names = {int(k): v for k, v in result.names.items()}
    order = np.argsort(probs)[::-1]
    top1_id, top2_id = int(order[0]), int(order[1])
    top1, top2 = names[top1_id], names[top2_id]
    score1, score2 = float(probs[top1_id]), float(probs[top2_id])
    margin = score1 - score2
    entropy = float(-np.sum(np.clip(probs, 1e-12, 1) * np.log(np.clip(probs, 1e-12, 1))) / np.log(len(probs)))
    decision = 'decisión clara' if score1 >= 0.75 and margin >= 0.25 else 'revisar'

    row = {
        'imagen': path.name,
        'clase_predicha': top1,
        'clase_predicha_es': SPANISH[top1],
        'puntaje_maximo': score1,
        'segunda_clase': top2,
        'segunda_clase_es': SPANISH[top2],
        'segundo_puntaje': score2,
        'margen_top1_top2': margin,
        'entropia_normalizada': entropy,
        'regla_revision': decision,
    }
    for class_id, probability in enumerate(probs):
        row[f'puntaje_{names[class_id]}'] = float(probability)
    rows.append(row)

predictions = pd.DataFrame(rows)
predictions.to_csv(RUN_RESULTS / 'predicciones_imagenes_nuevas.csv', index=False)

print(f'Tiempo total de inferencia: {elapsed:.3f} s')
print(f'Tiempo promedio: {elapsed * 1000 / len(uploaded_paths):.2f} ms/imagen')
display(predictions[[
    'imagen', 'clase_predicha_es', 'puntaje_maximo',
    'segunda_clase_es', 'margen_top1_top2', 'regla_revision'
]].round(4))


In [ ]:
# Figura individual optimizada para publicación: fotografía 65 % y gráfico 35 %.
for path, (_, row) in zip(uploaded_paths, predictions.iterrows()):
    with Image.open(path) as im:
        image = ImageOps.exif_transpose(im).convert('RGB')

    scores = [row[f'puntaje_{label}'] for label in CLASSES]
    labels_es = [SPANISH[label].capitalize() for label in CLASSES]
    colors = [COLORS[label] for label in CLASSES]

    fig, axes = plt.subplots(
        1, 2,
        figsize=(11.5, 6.2),
        gridspec_kw={'width_ratios': [1.65, 1.0]},
        constrained_layout=True,
    )
    fig.patch.set_facecolor('white')
    axes[0].imshow(image, aspect='equal')
    axes[0].axis('off')
    axes[0].set_anchor('C')
    axes[0].set_title('Imagen analizada', fontsize=12, fontweight='semibold', pad=10)

    bars = axes[1].barh(labels_es, scores, color=colors)
    axes[1].invert_yaxis()
    axes[1].set_xlim(0, 1.02)
    axes[1].set_xlabel('Puntaje softmax')
    axes[1].set_title('Distribución de puntajes', fontsize=12, fontweight='semibold', pad=38)
    axes[1].grid(axis='x', color='#d9d9d9', linewidth=0.7, alpha=0.8)
    axes[1].set_axisbelow(True)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    for bar, value in zip(bars, scores):
        x_text = value + 0.018 if value < 0.90 else value - 0.018
        ha = 'left' if value < 0.90 else 'right'
        text_color = '#222222' if value < 0.90 else 'white'
        axes[1].text(
            x_text, bar.get_y() + bar.get_height()/2,
            f'{value:.3f}', va='center', ha=ha,
            fontsize=9, color=text_color, fontweight='semibold'
        )
    status_color = '#1b7837' if row['regla_revision'] == 'decisión clara' else '#b2182b'
    axes[1].text(
        0.5, 1.075, row['regla_revision'].upper(),
        transform=axes[1].transAxes, ha='center', va='center',
        color='white', fontweight='bold', fontsize=10,
        bbox={'boxstyle': 'round,pad=0.4', 'facecolor': status_color, 'edgecolor': 'none'},
    )
    fig.suptitle(
        f"Clasificación de madurez · Predicción: {row['clase_predicha_es'].upper()}  "
        f"(puntaje {row['puntaje_maximo']:.3f}; margen {row['margen_top1_top2']:.3f})",
        fontsize=15, fontweight='bold'
    )

    output_path = FIGURES_DIR / f'{path.stem}_prediccion.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


In [ ]:
# Montaje resumen de hasta 12 imágenes para el libro.
n_show = min(12, len(uploaded_paths))
ncols = min(4, n_show)
nrows = math.ceil(n_show / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), squeeze=False)
fig.patch.set_facecolor('white')

for ax in axes.flat:
    ax.axis('off')

for ax, path, (_, row) in zip(axes.flat, uploaded_paths[:n_show], predictions.head(n_show).iterrows()):
    with Image.open(path) as im:
        image = ImageOps.exif_transpose(im).convert('RGB')
    ax.imshow(image, aspect='equal')
    ax.set_anchor('C')
    color = '#1b7837' if row['regla_revision'] == 'decisión clara' else '#b2182b'
    ax.set_title(
        f"{row['clase_predicha_es'].upper()} · {row['puntaje_maximo']:.2f}\n{row['regla_revision']}",
        color=color, fontsize=10, fontweight='bold'
    )
    ax.axis('off')

fig.suptitle('Clasificación de madurez de palma africana · YOLO11m', fontsize=15)
fig.tight_layout(rect=[0, 0, 1, 0.96])
montage_path = RUN_BOOK / 'montaje_predicciones.png'
fig.savefig(montage_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Montaje guardado:', montage_path)


In [ ]:
summary = {
    'run_id': RUN_ID,
    'model': 'YOLO11m-cls',
    'seed': 2026,
    'model_path': str(MODEL_PATH),
    'device': DEVICE_NAME,
    'n_images': len(uploaded_paths),
    'inference_total_seconds': round(elapsed, 6),
    'inference_ms_image': round(elapsed * 1000 / len(uploaded_paths), 4),
    'clear_decisions': int((predictions['regla_revision'] == 'decisión clara').sum()),
    'review_decisions': int((predictions['regla_revision'] == 'revisar').sum()),
    'threshold_rule': 'top1 >= 0.75 and top1-top2 >= 0.25',
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
}
(RUN_RESULTS / 'resumen_inferencia.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('NB05 FINALIZADO CORRECTAMENTE')
print('Predicciones:', RUN_RESULTS)
print('Figuras para el libro:', RUN_BOOK)


## Uso e interpretación

- Una salida `decisión clara` indica que el modelo separó ampliamente su primera opción de la segunda.
- Una salida `revisar` recomienda inspección humana, mejor fotografía o repetición de la captura.
- Los puntajes no deben interpretarse como certeza agronómica calibrada.
- El modelo fue desarrollado con un conjunto pequeño y posee pocas imágenes de las clases `damaged` y `empty`.
- Para uso de campo debe realizarse validación externa con imágenes nuevas y condiciones de captura distintas.
